In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


This notebook adds the dairy food groups to the SHeS 2021 diet data according to the disaggregation steps outlined in the study: https://doi.org/10.1016/j.cdnut.2024.103774

In [ ]:
%cd /content/drive/MyDrive/mSHIFT_SHeS/code_ocean/

/content/drive/MyDrive/mSHIFT_SHeS/code_ocean


In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import sys


In [ ]:
# Add the parent directory of the notebook to sys.path, enables module imports from analysis_code
sys.path.append(str(Path().resolve() / 'code/notebooks/notebook_code'))

In [ ]:
import data_processing
from data_processing import unique_mixed_items_dairy, initialise_dairy_mapping, complete_dairy_mapping, dairy_intake_diet_data, initialise_dairy_mapping_dictionary, initialise_ingredient_proportion, ingredient_weighting_item, ingredient_weighting_dict

In [ ]:
data_path = Path("data")

In [ ]:
## FSA recipe database
recipe_data = pd.read_excel(data_path / 'dairy_disag/NDB_SHeS_Disag_Dairy_08012024.xlsx')

# data on dairy content per food item
dairy_disag_data = pd.read_excel(data_path / 'dairy_disag/dairy_disag_16122023.xlsx')

# NDB 2022 data
ndb_data_path = data_path / 'NDB_data/NDB_intake24-nutrient-mapping-UK_V2_2022-18-10-2022.csv'
ndb_data = pd.read_csv(ndb_data_path)

In [ ]:
# Identify all food items in the NDB and the FSA recipe database that contain some dairy
dairy_codes_total = recipe_data[recipe_data['Dairy_g']>0]['IFoodCode'].unique().tolist()
dairy_codes_ndb = ndb_data[ndb_data['FCT record ID'].isin(dairy_codes_total)]['FCT record ID'].unique().tolist()

In [ ]:
 #manually change the food code for trifle to be consistent with SHeS
recipe_data.loc[recipe_data['L0FoodCode'] == 6964.0, 'L0FoodCode'] = 574

# rename code for CHOCOLATE CHIP CAKES MADE WITH PUFA MARGARINE, HOMEMADE to be consistent with SHeS
recipe_data.loc[recipe_data['L0FoodCode'] == 8606.0, 'L0FoodCode'] = 5201

# rename code for 'SPECIAL K CEREAL BARS, FRUIT WITH YOGURT TOPPING ONLY'
recipe_data.loc[recipe_data['L0FoodCode'] == 10187.0, 'L0FoodCode'] = 8136

# rename code for 'FLORENTINES'
recipe_data.loc[recipe_data['L0FoodCode'] == 330, 'L0FoodCode'] = 262

In [ ]:
# These categories are mutually exclusive. i.e. reductions at this level prevents double counting dairy reductions
dairy_categories = np.loadtxt(data_path / 'indicator_lists/food_groups_dairy.txt', dtype=str).tolist()

In [ ]:
dairy_map = initialise_dairy_mapping(food_groups=dairy_categories)
dairy_mapping = complete_dairy_mapping(dairy_map=dairy_map, dairy_disag_data=dairy_disag_data)

In [ ]:
# Inlcude additional dairy ingredient food codes present in the FSA recipe database but were not incuded in the NDB 2022

dairy_mapping['Milk_SemiSkimmed']['items_only'] += [608, 8543]
dairy_mapping['Milk_Skimmed']['items_only'] += [601, 613, 8544, 616]
dairy_mapping['Milk_Whole']['items_only'] += [9218, 602, 603, 604, 605]

dairy_mapping['Cheese_Skimmed']['items_only'] += [7738]
dairy_mapping['Cheese_SemiSkimmed']['items_only'] += [7112, 659, 10977, 7727]
dairy_mapping['Cheese_Whole']['items_only'] += [654, 688, 7735]

dairy_mapping['Cream_SemiSkimmed']['items_only'] += [639, 640]
dairy_mapping['Cream_Whole']['items_only'] += [645, 646]

with open('data/mappings/dairy_mapping.json', 'w') as fp:
    json.dump(dairy_mapping, fp)

In [ ]:
dairy_mapping_mixed_items = initialise_dairy_mapping_dictionary(dairy_categories=dairy_categories,
                                                                dairy_mapping=dairy_mapping,
                                                                recipe_data=recipe_data)

In [ ]:
# Nearest neighbour recipe assignment multiple codes for the same food item. Manually duplicate within the mapping.
# This reassignment follows the methods in dairy disag paper Jaacks et al, Disaggregation of Dairy in Composite Foods in the United Kingdom 2024

dairy_mapping_mixed_items['Milk_Skimmed'][11416] = dairy_mapping_mixed_items['Milk_Skimmed'][2276]
dairy_mapping_mixed_items['Milk_Skimmed'][11417] = dairy_mapping_mixed_items['Milk_Skimmed'][259]
dairy_mapping_mixed_items['Milk_Skimmed'][11419] = dairy_mapping_mixed_items['Milk_Skimmed'][268]

dairy_mapping_mixed_items['Milk_Whole'][11416] = dairy_mapping_mixed_items['Milk_Whole'][2276]
dairy_mapping_mixed_items['Milk_Whole'][11418] = dairy_mapping_mixed_items['Milk_Whole'][260]
dairy_mapping_mixed_items['Milk_Whole'][11420] = dairy_mapping_mixed_items['Milk_Whole'][269]
dairy_mapping_mixed_items['Milk_Whole'][11422] = dairy_mapping_mixed_items['Milk_Whole'][2254]

dairy_mapping_mixed_items['Butter'][11416] = dairy_mapping_mixed_items['Butter'][2276]

# assign the recipes which have an identical food code for dairy and non-dairy containing items to correspon to the dairy containing item
dairy_mapping_mixed_items['Butter'][5607] = dairy_mapping_mixed_items['Butter'][3955]
dairy_mapping_mixed_items['Butter'][1921] = dairy_mapping_mixed_items['Butter'][11602]

dairy_mapping_mixed_items['Cheese_Whole'][356] = dairy_mapping_mixed_items['Cheese_Whole'][821]
dairy_mapping_mixed_items['Cream_Whole'][356] = dairy_mapping_mixed_items['Cream_Whole'][821]
dairy_mapping_mixed_items['Milk_Skimmed'][356] = dairy_mapping_mixed_items['Milk_Skimmed'][821]

In [ ]:
# check that every mixed dairy item has been assigned dairy ingredients
for cat in dairy_categories:
  missing_codes=[]
  for i in dairy_mapping[cat]['items_mixed']:
    if i not in dairy_mapping_mixed_items[cat]:
      missing_codes.append(i)
  if len(missing_codes)>0:
    print(cat)
    print(missing_codes)
    print('\n')

for cat in dairy_categories:
  print(cat, len(dairy_mapping_mixed_items[cat]), len(dairy_mapping[cat]['items_mixed']))

Milk_Skimmed 316 315
Milk_SemiSkimmed 116 116
Milk_Whole 128 128
Cheese_Skimmed 4 4
Cheese_SemiSkimmed 38 38
Cheese_Whole 148 147
Yogurt_Skimmed 5 5
Yogurt_SemiSkimmed 19 19
Yogurt_Whole 29 29
Cream_SemiSkimmed 60 60
Cream_Whole 157 156
Butter 191 191


In [ ]:
# Save the mapping
filename = data_path / 'dairy_ingredients_dict.json'

with open(filename, 'w') as f:
    json.dump(dairy_mapping_mixed_items, f, indent=4)

In [ ]:
# Rename for consistent naming conventions of ingredient dictionary
dairy_mapping_dict = dairy_mapping_mixed_items.copy()

In [ ]:
diet_data = pd.read_parquet(data_path / "diet_data.parquet")

In [ ]:
# Initialise the dairy categories
diet_data.loc[:, dairy_categories] = np.nan

In [ ]:
# Include the disaggregated dairy contribution in the diet data
for cat in dairy_categories:
  diet_data = diet_data.apply(lambda row: dairy_intake_diet_data(row, category=cat,
                                                                 diet_data=diet_data,
                                                                dairy_per100g=dairy_disag_data,
                                                                dairy_codes=dairy_codes_total),
                               axis=1)

In [ ]:
# Save the item level SHeS data that includes dairy
save_path = data_path / "diet_data.parquet"
diet_data.to_parquet(save_path)

In [ ]:
# Produce the dictionary mapping dairy containing items to a dictionary of the proportional contribution each dairy inrgedient makes to the dairy content of each dairy food group
dairy_ingredient_proportion_dict = ingredient_weighting_dict(ingredient_dict=dairy_mapping_dict, recipe_data=recipe_data)

Copy the proportional contribution of whole cheese for 'Savoury pastry (e.g. cheese pastry)' which only exists in SHeS with that for 'Cheese and onion pasty/roll (includes potato)'

In [ ]:
dairy_ingredient_proportion_dict['Milk_Skimmed']['11416'] = dairy_ingredient_proportion_dict['Milk_Skimmed'][2276]
dairy_ingredient_proportion_dict['Milk_Skimmed']['11417'] = dairy_ingredient_proportion_dict['Milk_Skimmed'][259]
dairy_ingredient_proportion_dict['Milk_Skimmed']['11419'] = dairy_ingredient_proportion_dict['Milk_Skimmed'][268]
dairy_ingredient_proportion_dict['Cheese_Whole']['356'] = dairy_ingredient_proportion_dict['Cheese_Whole'][821.0]

In [ ]:
# Save the mapping
file_path = data_path / 'dairy_ingredient_proportion_dict.json'

with open(file_path, 'w') as f:
    json.dump(dairy_ingredient_proportion_dict, f, indent=4)

In [ ]:
# check the mapping loads properly
with open(file_path, 'r') as f:
    dairy_ingredient_proportion_dict = json.load(f)